# Machine Learning Prediction of Postoperative Pain Improvement after CABG

**Companion code for the article:** *Machine Learning Prediction of Postoperative Pain Improvement
Following Coronary Artery Bypass Grafting Using Preoperative Clinical, Psychosocial, Fatigue, and
Functional Factors* (Mara'Beh R.A., Mara'Beh S., Sawalha O.).

This notebook reproduces every analysis reported in the manuscript, from data preprocessing to the
robustness analyses added during peer review.

**Data are not included.** The patient-level dataset cannot be shared publicly. It is available from
the authors on reasonable request (see the article's *Data availability* statement). To run the
notebook, save the dataset as `Data.xlsx` in the same folder as this notebook, or change `DATA_PATH`
below. Any dataset with the same column names and codings can be used (see `feature_dictionary.csv`
and `input_template.xlsx` in this repository), for example to externally validate the model.

**Privacy note.** Cells that would display individual patient records have been intentionally
omitted. Please do not publish a version of this notebook containing outputs generated from the
real dataset, because some outputs may reveal patient-level information.

## Contents
1. Setup and configuration
2. Data loading and predictor definitions
3. Encoding of categorical variables
4. Missing data
5. Outcome definition
6. Descriptive univariate analysis (not used for feature selection)
7. Train/test split and hyperparameter tuning
8. Model comparison on the independent test set
9. Final model: confusion matrix, ROC curve and bootstrap confidence intervals
10. Explainability: permutation importance and SHAP
11. Calibration and decision curve analysis
12. Outcome construction, ceiling effect and sensitivity analyses
13. Hospital-level (leave-one-hospital-out) validation
14. Repeated cross-validation and statistical comparison of models
15. Pain-only benchmarks and the subgroup without severe baseline pain
16. Decision-threshold analysis
17. Correlated predictors and domain-level importance
18. Overfitting and stability (nested CV, permutation test, learning curve)
19. Alternative outcome and leave-one-hospital-out calibration / decision curves
20. External validation helper and reproducibility summary

## 1. Setup and configuration

In [ ]:
# ---------------------------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------------------------
import sys, platform, warnings
warnings.filterwarnings('ignore')            # hide library deprecation warnings for readability

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import statsmodels
import statsmodels.api as sm

# scikit-learn building blocks
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import (train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
                                     GridSearchCV, cross_validate, cross_val_predict,
                                     LeaveOneGroupOut, permutation_test_score, learning_curve)
# the six classifiers compared in the study
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
# evaluation metrics and tools
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score, matthews_corrcoef,
                             roc_auc_score, roc_curve, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
# statistical tests
from scipy.stats import mannwhitneyu, spearmanr
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import shap                                    # SHapley Additive exPlanations

# ---------------------------------------------------------------------------------------------
# Configuration -- edit these values if needed
# ---------------------------------------------------------------------------------------------
DATA_PATH = 'Data.xlsx'      # path to the (non-public) dataset
RANDOM_STATE = 42            # single fixed seed used for every random step (splits, CV, models)
SAVE_OUTPUTS = True          # if True, tables are written to .xlsx and figures to .png in the working folder

# The robustness analyses in Sections 18-19 refit hundreds of grid searches and can take
# 30-60 minutes on a laptop. Set RUN_SLOW_ANALYSES = False to skip them.
RUN_SLOW_ANALYSES = True
N_NESTED_REPEATS = 5         # repetitions of the outer 5-fold loop in nested CV (Section 18)
N_PERMUTATIONS = 1000        # label permutations for the permutation tests (Section 18)
N_BOOTSTRAP = 2000           # bootstrap resamples for test-set confidence intervals (Section 9)

rng = np.random.RandomState(RANDOM_STATE)    # random generator used for all bootstrap resampling

# Print software versions so that results can be matched to the environment that produced them.
print("Python:", sys.version.split()[0], "|", platform.platform())
for name, mod in [('pandas', pd), ('numpy', np), ('scipy', scipy), ('scikit-learn', sklearn),
                  ('statsmodels', statsmodels), ('matplotlib', matplotlib), ('shap', shap)]:
    print(f"{name}: {mod.__version__}")
# The manuscript results were produced with Python 3.11.15, pandas 3.0.2, numpy 2.4.4, scipy 1.17.1,
# scikit-learn 1.8.0, statsmodels 0.14.6 and shap 0.51.0 (see requirements.txt).


def save_table(df, filename, **kwargs):
    """Write a results table to Excel when SAVE_OUTPUTS is True."""
    if SAVE_OUTPUTS:
        df.to_excel(filename, **kwargs)


def save_figure(filename):
    """Save the current matplotlib figure at publication resolution when SAVE_OUTPUTS is True."""
    if SAVE_OUTPUTS:
        plt.savefig(filename, dpi=300, bbox_inches='tight')

## 2. Data loading and predictor definitions

Only variables measured **before surgery** are used as predictors, so that the model could be applied
at the preoperative assessment. The 37 candidate predictors were pre-specified from the literature and
were all retained (no data-driven feature selection was performed).

In [ ]:
# Read the Excel file (one row per patient).
data = pd.read_excel(DATA_PATH)
print("Dataset dimensions (patients, columns):", data.shape)   # only the size is printed, not the records

# The 37 pre-specified preoperative predictors, grouped by domain.
# Column names are those of the source dataset; see feature_dictionary.csv for full definitions.
feature_columns = [
    # Demographic
    'AGE_years', 'wieght', 'Height', 'gender', 'MaritalStatus',
    'numberofchilderen', 'educationlevel', 'occupation',
    # Clinical and lifestyle
    'HTN', 'Smoke', 'Alcohol', 'Medication_reccd', 'economicproblems',
    # Preoperative pain category (1 = no pain, 2 = mild, 3 = moderate, 4 = severe)
    'prepainscaletotal2023',
    # Psychological state (DASS-21 subscales)
    'Depresion_pre', 'Ansiedad_pre', 'Estres_pre',
    # Physical fitness (International Fitness Scale)
    'preTotalIFIS',
    # Fatigue (Fatigue Scale for Motor and Cognitive Functions)
    'prefatiguecognitivescale', 'prefatiguephysicalscale', 'preFSMCtotal',
    # Comorbidity: Charlson index and the age band used for its age adjustment
    'precharolsonindex', 'AgeCharlson',
    # Karnofsky Performance Status
    'prekps',
    # Family functioning (Family APGAR)
    'preapgar',
    # Functional Independence Measure (FIM) domains and summary scores
    'preselfcare', 'preSphinctercontrol', 'pretransfer', 'prelocomtion',
    'PRPMotorScore', 'precommunication', 'presocialcogintive',
    'PRPCognitiveScore', 'PRPTOTALFIM',
    # Psychological state of the family member / companion (DASS-21 subscales)
    'prefamiliaestres', 'prefamiliaresdepreseion', 'preansiedadfamilia'
]

PAIN = 'prepainscaletotal2023'          # short name for the preoperative pain predictor, used repeatedly
target_column = 'pain_change'           # postoperative pain category minus preoperative pain category

X_raw = data[feature_columns].copy()    # predictors before encoding
y_numeric_all = data[target_column].copy()
print("Predictor matrix:", X_raw.shape, "| outcome vector:", y_numeric_all.shape)

# Readable labels used in tables and figures.
feature_display_names = {
    'AGE_years': 'Age (Years)', 'wieght': 'Weight', 'Height': 'Height', 'gender': 'Gender',
    'MaritalStatus': 'Marital Status', 'numberofchilderen': 'Number of Children',
    'educationlevel': 'Education Level', 'occupation': 'Occupation', 'HTN': 'Hypertension',
    'Smoke': 'Smoking Status', 'Alcohol': 'Alcohol Consumption', 'Medication_reccd': 'Medication Received',
    'economicproblems': 'Economic Problems', 'prepainscaletotal2023': 'Preoperative Pain Score',
    'Depresion_pre': 'Preoperative Depression', 'Ansiedad_pre': 'Preoperative Anxiety',
    'Estres_pre': 'Preoperative Stress', 'preTotalIFIS': 'Preoperative IFIS Score',
    'prefatiguecognitivescale': 'Preoperative Cognitive Fatigue',
    'prefatiguephysicalscale': 'Preoperative Physical Fatigue', 'preFSMCtotal': 'Preoperative FSMC Total Score',
    'precharolsonindex': 'Preoperative Charlson Index', 'AgeCharlson': 'Age-Adjusted Charlson Index',
    'prekps': 'Preoperative Karnofsky Performance Score', 'preapgar': 'Preoperative APGAR Score',
    'preselfcare': 'Preoperative Self-Care', 'preSphinctercontrol': 'Preoperative Sphincter Control',
    'pretransfer': 'Preoperative Transfer Ability', 'prelocomtion': 'Preoperative Locomotion',
    'PRPMotorScore': 'Preoperative Motor FIM Score', 'precommunication': 'Preoperative Communication',
    'presocialcogintive': 'Preoperative Social Cognition', 'PRPCognitiveScore': 'Preoperative Cognitive FIM Score',
    'PRPTOTALFIM': 'Preoperative Total FIM Score', 'prefamiliaestres': 'Family Stress',
    'prefamiliaresdepreseion': 'Family Depression', 'preansiedadfamilia': 'Family Anxiety'
}

## 3. Encoding of categorical variables

Text categories are converted to numbers with **fixed, pre-defined mappings**. Because these mappings
do not use the outcome or statistics estimated from other patients, applying them to the whole dataset
before splitting does not cause information leakage.

In [ ]:
def encode_features(X_in):
    """Convert text-valued predictor columns to numeric codes using fixed mappings.

    The function returns a copy and never looks at the outcome, so it is safe to apply to the
    full dataset and to any external dataset with the same columns.
    """
    X = X_in.copy()
    # Binary: male = 1, female = 0 (both capitalisations accepted)
    X['gender'] = X['gender'].map({'Male': 1, 'Female': 0, 'male': 1, 'female': 0})
    # Nominal: single/bachelor = 0, married = 1, divorced = 2, widowed = 3
    X['MaritalStatus'] = X['MaritalStatus'].map({
        'Married': 1, 'Single': 0, 'Bachelor': 0, 'Divorced': 2, 'Widow': 3
    })
    # Ordinal: higher code = higher educational attainment
    X['educationlevel'] = X['educationlevel'].map({
        'Primary': 0, 'Secondary': 1, 'Diploma': 2,
        'Professional education/high school': 3, 'High school': 4,
        'University degree': 5, 'University': 5
    })
    # Occupation recorded as free text: housewife or none = 0, any other occupation = 1
    X['occupation'] = X['occupation'].astype(str).str.strip().str.lower()
    X['occupation'] = X['occupation'].apply(lambda v: 0 if v in ['housewife', 'house wife', 'none'] else 1)
    # Yes/no items: yes = 1, no = 0
    for col in ['HTN', 'Smoke', 'Alcohol', 'Medication_reccd', 'economicproblems']:
        X[col] = X[col].map({'Yes': 1, 'No': 0, 'yes': 1, 'no': 0})
    # Age band used for the Charlson age adjustment, encoded by its lower bound (years)
    X['AgeCharlson'] = X['AgeCharlson'].map({'80-89': 80, '70-79': 70, '60-69': 60, '50-59': 50})
    return X


X_encoded = encode_features(X_raw)
# Quick sanity check: every column should now be numeric.
print("Non-numeric columns after encoding:",
      [c for c in X_encoded.columns if not pd.api.types.is_numeric_dtype(X_encoded[c])])

## 4. Missing data

Missingness was restricted to one predictor (the Charlson index, 8 of 200 patients). The main analysis
uses complete cases (N = 192). As a sensitivity analysis, the missing values are imputed with a
k-nearest-neighbour imputer placed **inside** the cross-validation pipeline, so that the imputer is
fitted only on the training folds.

In [ ]:
# Count missing values per predictor.
missing_counts = X_encoded.isna().sum()
print("Predictors with missing values:\n", missing_counts[missing_counts > 0])
print("Patients with any missing predictor value:", X_encoded.isna().any(axis=1).sum())

# Complete-case dataset used for all main analyses (exclusion depends only on predictor missingness).
complete_mask = X_encoded['precharolsonindex'].notna()
X = X_encoded[complete_mask].reset_index(drop=True)
y_numeric = y_numeric_all[complete_mask].reset_index(drop=True)
print("Complete-case dataset:", X.shape)


def to_outcome_class(y_num):
    """Map the numeric pain change to the two outcome classes (defined in Section 5)."""
    return pd.Series(np.where(y_num < 0, 'Improved', 'Not Improved'), name='outcome')


# Sensitivity analysis: keep all 200 patients and impute inside each CV training fold.
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
imputation_pipe = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),   # imputes using the 5 most similar training patients
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=RANDOM_STATE))
])
complete_pipe = Pipeline(imputation_pipe.steps[1:])   # same model without the imputer

for label, Xd, yd, pipe in [('Complete case (N=192)', X, to_outcome_class(y_numeric), complete_pipe),
                            ('KNN-imputed  (N=200)', X_encoded, to_outcome_class(y_numeric_all), imputation_pipe)]:
    sc = cross_validate(pipe, Xd, yd, cv=cv5, scoring={'f1': 'f1_weighted', 'acc': 'accuracy'})
    print(f"{label}: CV weighted F1 = {sc['test_f1'].mean():.3f}, CV accuracy = {sc['test_acc'].mean():.3f}")

## 5. Outcome definition

`pain_change` = postoperative pain category − preoperative pain category (four ordered categories).
A patient is **Improved** if the category decreased (`pain_change < 0`) and **Not Improved** otherwise
(no change or worsening).

In [ ]:
y = to_outcome_class(y_numeric)
print(y.value_counts())                         # absolute class sizes
print(y.value_counts(normalize=True).round(3))  # class proportions

## 6. Descriptive univariate analysis

For each predictor, a Mann–Whitney U test (Improved vs Not Improved) and the mutual information with the
outcome are computed. **This table is descriptive only and was not used to select predictors**; all 37
predictors enter every model.

In [ ]:
y_bin = (y_numeric < 0).astype(int)    # 1 = Improved, 0 = Not Improved

rows = []
for col in X.columns:
    g1, g0 = X.loc[y_bin == 1, col], X.loc[y_bin == 0, col]
    try:
        _, p_value = mannwhitneyu(g1, g0, alternative='two-sided')
    except ValueError:                 # raised for a constant column (e.g. alcohol in this cohort)
        p_value = np.nan
    rows.append({'Feature': feature_display_names[col], 'Mann-Whitney p-value': p_value})
univariate_table = pd.DataFrame(rows)
univariate_table['Mutual Information'] = mutual_info_classif(X, y_bin, random_state=RANDOM_STATE)
univariate_table = univariate_table.sort_values('Mutual Information', ascending=False).reset_index(drop=True)
save_table(univariate_table, 'feature_selection_justification.xlsx', index=False)
univariate_table.head(10).round(3)

In [ ]:
# Leakage check: recompute the mutual-information ranking using only the training folds and compare
# it with the full-sample ranking. (The split itself is created in Section 7; it is repeated here so
# that this cell can be read on its own.)
_X_tr, _, _y_tr, _ = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)
full_rank = pd.Series(mutual_info_classif(X, y_bin, random_state=RANDOM_STATE), index=X.columns).rank(ascending=False)
fold_ranks = []
for tr_idx, _va in cv5.split(_X_tr, _y_tr):
    mi = mutual_info_classif(_X_tr.iloc[tr_idx], (_y_tr.iloc[tr_idx] == 'Improved').astype(int),
                             random_state=RANDOM_STATE)
    fold_ranks.append(pd.Series(mi, index=X.columns).rank(ascending=False))
fold_ranks = pd.concat(fold_ranks, axis=1)
rho, p = spearmanr(full_rank, fold_ranks.mean(axis=1))
print(f"Full-sample vs training-fold MI ranking: Spearman rho = {rho:.3f}, p = {p:.2g}")

## 7. Train/test split and hyperparameter tuning

* Stratified 70/30 split; the 30% test set is **never used for tuning**.
* Standardisation is placed inside a `Pipeline` for the scale-sensitive models (Logistic Regression,
  SVC, KNN), so scaling parameters are learned from training folds only.
* Each model is tuned with an exhaustive `GridSearchCV` (stratified 5-fold CV on the training set,
  weighted F1-score as the selection criterion).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)   # stratify keeps class proportions
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)   # CV used for tuning
print("Training set:", X_train.shape, "| test set:", X_test.shape)
print("Training classes:", y_train.value_counts().to_dict(), "| test classes:", y_test.value_counts().to_dict())

# Each entry: (estimator or pipeline, hyperparameter grid). In a Pipeline, 'clf__C' means
# "parameter C of the step named clf".
model_grids = {
    "Logistic Regression": (
        Pipeline([('scaler', StandardScaler()),
                  ('clf', LogisticRegression(max_iter=5000, class_weight='balanced', random_state=RANDOM_STATE))]),
        {'clf__C': [0.01, 0.1, 1, 10, 100]}                               # inverse regularisation strength
    ),
    "SVC": (
        Pipeline([('scaler', StandardScaler()),
                  ('clf', SVC(probability=True, random_state=RANDOM_STATE))]),   # probability=True -> Platt scaling
        {'clf__C': [0.01, 0.1, 1, 10, 100],                              # margin penalty
         'clf__gamma': ['scale', 0.001, 0.01, 0.1, 1],                    # RBF kernel width
         'clf__kernel': ['rbf', 'linear'],
         'clf__class_weight': [None, 'balanced']}
    ),
    "KNN": (
        Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]),
        {'clf__n_neighbors': [3, 5, 7, 9, 11],
         'clf__weights': ['uniform', 'distance'],
         'clf__p': [1, 2]}                                                # 1 = Manhattan, 2 = Euclidean distance
    ),
    "Decision Tree": (
        DecisionTreeClassifier(class_weight='balanced', random_state=RANDOM_STATE),
        {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10], 'min_samples_leaf': [1, 2, 4]}
    ),
    "Random Forest": (
        RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
        {'n_estimators': [100, 200, 300], 'max_depth': [None, 5, 10], 'min_samples_leaf': [1, 2, 4]}
    ),
    "Gradient Boosting": (
        GradientBoostingClassifier(random_state=RANDOM_STATE),
        {'n_estimators': [100, 200], 'learning_rate': [0.01, 0.05, 0.1], 'max_depth': [2, 3, 4]}
    ),
}

best_models, tuning_rows = {}, []
for name, (estimator, grid) in model_grids.items():
    gs = GridSearchCV(estimator, grid, cv=cv, scoring='f1_weighted', n_jobs=-1)  # n_jobs=-1 uses all CPU cores
    gs.fit(X_train, y_train)                       # tuning uses the training set only
    best_models[name] = gs.best_estimator_         # refitted on the full training set with the best parameters
    tuning_rows.append({'Model': name, 'Best CV weighted F1': gs.best_score_, 'Best parameters': gs.best_params_})
    print(f"{name:20s} best CV F1 = {gs.best_score_:.3f}  {gs.best_params_}")
tuning_summary = pd.DataFrame(tuning_rows)
save_table(tuning_summary, 'hyperparameter_tuning_summary.xlsx', index=False)

## 8. Model comparison on the independent test set

In [ ]:
def p_improved(model, Xd):
    """Predicted probability of the 'Improved' class (column order follows model.classes_)."""
    return model.predict_proba(Xd)[:, list(model.classes_).index('Improved')]


y_test_bin = (y_test == 'Improved').astype(int)    # 1 = Improved, used for ROC-AUC
results = []
for name, model in best_models.items():
    # 5-fold CV estimate on the training set (same folds as tuning)
    cv_res = cross_validate(model, X_train, y_train, cv=cv,
                            scoring={'acc': 'accuracy', 'f1': 'f1_weighted', 'bal_acc': 'balanced_accuracy'})
    model.fit(X_train, y_train)                    # final fit on the whole training set
    y_pred_m = model.predict(X_test)               # class predictions for the untouched test set
    results.append({
        'Model': name,
        'CV Accuracy': cv_res['test_acc'].mean(), 'CV F1': cv_res['test_f1'].mean(),
        'Test Accuracy': accuracy_score(y_test, y_pred_m),
        'Test F1': f1_score(y_test, y_pred_m, average='weighted'),
        'Test Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_m),
        'Test MCC': matthews_corrcoef(y_test, y_pred_m),
        'Test ROC-AUC': roc_auc_score(y_test_bin, p_improved(model, X_test)),
    })
results_df = pd.DataFrame(results).sort_values('Test F1', ascending=False).reset_index(drop=True)
save_table(results_df, 'model_comparison_results.xlsx', index=False)
results_df.round(3)

## 9. Final model: confusion matrix, ROC curve and bootstrap confidence intervals

The SVC is used as the final model. Note: the choice among the six algorithms also took test-set
results into account; Section 18 quantifies the resulting optimism with a nested cross-validation in
which algorithm selection is repeated inside every training fold.

In [ ]:
best_model_name = 'SVC'
best_model = best_models[best_model_name]
y_pred = best_model.predict(X_test)            # SVM decision-boundary predictions (no probability cut-off)
print(classification_report(y_test, y_pred))

# "Improved" is the positive class: sensitivity = recall of Improved, specificity = recall of Not Improved.
cm = confusion_matrix(y_test, y_pred, labels=['Improved', 'Not Improved'])
tp, fn, fp, tn = cm.ravel()
sensitivity, specificity = tp / (tp + fn), tn / (tn + fp)
print(f"Sensitivity = {sensitivity:.3f}, specificity = {specificity:.3f}, "
      f"PPV = {tp / (tp + fp):.3f}, NPV = {tn / (tn + fn):.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
ConfusionMatrixDisplay(cm, display_labels=['Improved', 'Not Improved']).plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title('Confusion Matrix - SVC')
plt.tight_layout(); save_figure('confusion_matrix.png'); plt.show()

In [ ]:
# ROC curve of the final model.
y_prob_pos = p_improved(best_model, X_test)
fpr, tpr, _ = roc_curve(y_test_bin, y_prob_pos)
auc_value = roc_auc_score(y_test_bin, y_prob_pos)
plt.figure(figsize=(7, 7))
plt.plot(fpr, tpr, lw=2, label=f'SVC (AUC = {auc_value:.3f})')
plt.plot([0, 1], [0, 1], ls='--', lw=1)           # chance diagonal
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve')
plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout()
save_figure('ROC.png'); plt.show()

In [ ]:
# Percentile bootstrap 95% CIs: resample the 58 test patients with replacement N_BOOTSTRAP times and
# recompute each metric on every resample.
y_test_arr = y_test.values
boot = {k: [] for k in ['Accuracy', 'Weighted F1', 'Balanced accuracy', 'MCC', 'ROC-AUC', 'Sensitivity', 'Specificity']}
idx = np.arange(len(y_test_arr))
for _ in range(N_BOOTSTRAP):
    b = rng.choice(idx, size=len(idx), replace=True)
    yt, yp, pr = y_test_arr[b], y_pred[b], y_prob_pos[b]
    if len(set(yt)) < 2:                           # skip resamples containing only one class
        continue
    boot['Accuracy'].append(accuracy_score(yt, yp))
    boot['Weighted F1'].append(f1_score(yt, yp, average='weighted'))
    boot['Balanced accuracy'].append(balanced_accuracy_score(yt, yp))
    boot['MCC'].append(matthews_corrcoef(yt, yp))
    boot['ROC-AUC'].append(roc_auc_score((yt == 'Improved').astype(int), pr))
    c = confusion_matrix(yt, yp, labels=['Improved', 'Not Improved'])
    boot['Sensitivity'].append(c[0, 0] / c[0].sum())
    boot['Specificity'].append(c[1, 1] / c[1].sum())

point = {'Accuracy': accuracy_score(y_test, y_pred), 'Weighted F1': f1_score(y_test, y_pred, average='weighted'),
         'Balanced accuracy': balanced_accuracy_score(y_test, y_pred), 'MCC': matthews_corrcoef(y_test, y_pred),
         'ROC-AUC': auc_value, 'Sensitivity': sensitivity, 'Specificity': specificity}
boot_ci = pd.DataFrame([{'Metric': k, 'Point estimate': point[k],
                         '95% CI lower': np.percentile(v, 2.5), '95% CI upper': np.percentile(v, 97.5)}
                        for k, v in boot.items()])
save_table(boot_ci, 'bootstrap_confidence_intervals.xlsx', index=False)
boot_ci.round(3)

## 10. Explainability: permutation importance and SHAP

In [ ]:
# Permutation importance: shuffle one predictor at a time in the test set and measure the drop in
# weighted F1-score (30 shuffles per predictor).
perm = permutation_importance(best_model, X_test, y_test, n_repeats=30,
                              random_state=RANDOM_STATE, scoring='f1_weighted')
importance_df = pd.DataFrame({'Feature': [feature_display_names[c] for c in X.columns],
                              'Importance': perm.importances_mean}).sort_values('Importance', ascending=False)
top = importance_df.head(15)
plt.figure(figsize=(10, 7))
plt.barh(top['Feature'][::-1], top['Importance'][::-1])   # reversed so the largest bar is on top
plt.xlabel('Mean decrease in weighted F1-score'); plt.title('Permutation Importance - SVC')
plt.tight_layout(); save_figure('feature_importance_plot.png'); plt.show()

In [ ]:
# Kernel SHAP explains the predicted probability of 'Improved' for each test patient.
# KernelExplainer is model-agnostic; it needs a function of a NumPy array and a background sample.
def predict_improved_proba(arr):
    return p_improved(best_model, pd.DataFrame(arr, columns=X_train.columns))

background = shap.sample(X_train, 30, random_state=RANDOM_STATE)   # 30 training patients as reference
explainer = shap.KernelExplainer(predict_improved_proba, background)
shap_values = explainer.shap_values(X_test, nsamples=100)         # Monte Carlo estimate (varies slightly by run)

X_test_disp = X_test.rename(columns=feature_display_names)
shap.summary_plot(shap_values, X_test_disp, show=False, plot_size=(9, 7))
plt.title('SHAP Summary - SVC'); plt.tight_layout(); save_figure('shap_summary_plot.png'); plt.show()

mean_abs_shap = pd.DataFrame({'Feature': X_test_disp.columns,
                              'Mean |SHAP value|': np.abs(shap_values).mean(axis=0)}
                             ).sort_values('Mean |SHAP value|', ascending=False)
save_table(mean_abs_shap, 'shap_feature_importance.xlsx', index=False)

# Agreement between the two rankings across all predictors and among the top 10.
merged = importance_df.merge(mean_abs_shap, on='Feature')
rho, p = spearmanr(merged['Importance'], merged['Mean |SHAP value|'])
overlap = set(importance_df.Feature[:10]) & set(mean_abs_shap.Feature[:10])
print(f"Permutation vs SHAP ranking: Spearman rho = {rho:.3f} (p = {p:.3f}); top-10 overlap = {len(overlap)}: {sorted(overlap)}")

## 11. Calibration and decision curve analysis

Both analyses use the clinically actionable risk: the predicted probability of **not improving**.

In [ ]:
p_not_test = 1 - y_prob_pos                         # predicted risk of not improving
y_not_test = (y_test == 'Not Improved').astype(int).values
brier = brier_score_loss(y_not_test, p_not_test)
frac_pos, mean_pred = calibration_curve(y_not_test, p_not_test, n_bins=5, strategy='quantile')


def net_benefit(y_true, prob, t):
    """Net benefit of treating patients whose predicted risk >= threshold t (Vickers & Elkin, 2006)."""
    n = len(y_true)
    tp_ = np.sum((prob >= t) & (y_true == 1)); fp_ = np.sum((prob >= t) & (y_true == 0))
    return tp_ / n - fp_ / n * t / (1 - t)


thresholds = np.linspace(0.05, 0.95, 37)
prev = y_not_test.mean()
dca_df = pd.DataFrame({'Threshold': thresholds,
                       'NB model': [net_benefit(y_not_test, p_not_test, t) for t in thresholds],
                       'NB treat all': [prev - (1 - prev) * t / (1 - t) for t in thresholds],
                       'NB treat none': 0.0})

# Bootstrap bands (1,000 resamples) for the calibration curve, Brier score and net benefit.
nb_boot, brier_boot, cal_boot = [], [], []
for _ in range(1000):
    b = rng.choice(len(y_not_test), len(y_not_test), replace=True)
    if len(set(y_not_test[b])) < 2:
        continue
    nb_boot.append([net_benefit(y_not_test[b], p_not_test[b], t) for t in thresholds])
    brier_boot.append(brier_score_loss(y_not_test[b], p_not_test[b]))
    fp_b, _ = calibration_curve(y_not_test[b], p_not_test[b], n_bins=5, strategy='quantile')
    if len(fp_b) == len(frac_pos):
        cal_boot.append(fp_b)
dca_df['NB model CI low'] = np.percentile(nb_boot, 2.5, axis=0)
dca_df['NB model CI high'] = np.percentile(nb_boot, 97.5, axis=0)
save_table(dca_df, 'decision_curve_analysis_with_ci.xlsx', index=False)
print(f"Brier score = {brier:.3f} (95% CI {np.percentile(brier_boot, 2.5):.3f}-{np.percentile(brier_boot, 97.5):.3f})")

# Threshold ranges in which the model beats both default strategies.
beats = (dca_df['NB model'] > dca_df['NB treat all']) & (dca_df['NB model'] > 0)
beats_ci = (dca_df['NB model CI low'] > dca_df['NB treat all']) & (dca_df['NB model CI low'] > 0)
print("Model beats both defaults (point estimate):", dca_df.Threshold[beats].min().round(3), "-", dca_df.Threshold[beats].max().round(3))
print("Model beats both defaults (lower 95% bound):", dca_df.Threshold[beats_ci].min().round(3), "-", dca_df.Threshold[beats_ci].max().round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
ax = axes[0]
ax.plot(mean_pred, frac_pos, 'o-', label=f'SVC (Brier = {brier:.3f})')
ax.fill_between(mean_pred, np.percentile(cal_boot, 2.5, axis=0), np.percentile(cal_boot, 97.5, axis=0),
                alpha=0.2, label='95% bootstrap CI')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability of not improving'); ax.set_ylabel('Observed fraction not improved')
ax.set_title('(a) Calibration curve (risk of not improving)'); ax.legend(loc='upper left'); ax.grid(alpha=0.3)
ax = axes[1]
ax.plot(dca_df.Threshold, dca_df['NB model'], lw=2, label='SVC')
ax.fill_between(dca_df.Threshold, dca_df['NB model CI low'], dca_df['NB model CI high'], alpha=0.2, label='95% bootstrap CI')
ax.plot(dca_df.Threshold, dca_df['NB treat all'], '--', label='Treat all patients')
ax.axhline(0, ls=':', color='C2', label='Treat none')
ax.set_ylim(-0.1, 0.6); ax.set_xlabel('Threshold probability'); ax.set_ylabel('Net benefit')
ax.set_title('(b) Decision curve analysis (risk of not improving)'); ax.legend(loc='upper right'); ax.grid(alpha=0.3)
plt.tight_layout(); save_figure('calibration_dca.png'); plt.show()

## 12. Outcome construction, ceiling effect and sensitivity analyses

Because the outcome is calculated from the preoperative pain category, which is also a predictor,
patients with severe (top-category) baseline pain can only stay the same or improve. These analyses
quantify how much of the performance depends on that structure.

In [ ]:
ceiling = (X[PAIN] == 4).values     # True for patients with severe preoperative pain
post_cat = data.loc[complete_mask, 'postpainscaletotal2023'].reset_index(drop=True)
print("Preoperative pain categories:", X[PAIN].value_counts().sort_index().to_dict())
print("Severe-baseline patients:", ceiling.sum(), "| still severe after surgery:", (post_cat[ceiling] == 4).sum())
ceiling_tab = pd.crosstab(np.where(ceiling, 'Severe baseline', 'Mild/moderate baseline'), y)
save_table(ceiling_tab, 'ceiling_effect_crosstab.xlsx')
ceiling_tab

In [ ]:
# Sensitivity analysis 1: re-tune and re-evaluate the SVC with the preoperative pain predictor removed.
svc_pipe, svc_grid = model_grids['SVC']
gs_nopain = GridSearchCV(svc_pipe, svc_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
gs_nopain.fit(X_train.drop(columns=[PAIN]), y_train)
model_nopain = gs_nopain.best_estimator_
pred_np = model_nopain.predict(X_test.drop(columns=[PAIN]))
print("Without preoperative pain -> test accuracy %.3f, balanced accuracy %.3f, ROC-AUC %.3f" % (
    accuracy_score(y_test, pred_np), balanced_accuracy_score(y_test, pred_np),
    roc_auc_score(y_test_bin, p_improved(model_nopain, X_test.drop(columns=[PAIN])))))

In [ ]:
# Sensitivity analysis 2: a SUBGROUP-SPECIFIC model, trained and tuned only on the 122 patients
# without severe baseline pain. Its held-out split contains different patients from the subgroup
# patients of the main test set, so its results are not directly comparable with Section 15 (i).
X_nc, y_nc = X[~ceiling].reset_index(drop=True), y[~ceiling].reset_index(drop=True)
Xnc_tr, Xnc_te, ync_tr, ync_te = train_test_split(X_nc, y_nc, test_size=0.30, random_state=RANDOM_STATE, stratify=y_nc)
model_nc = GridSearchCV(svc_pipe, svc_grid, cv=cv, scoring='f1_weighted', n_jobs=-1).fit(Xnc_tr, ync_tr).best_estimator_

# Primary estimate: repeated 5-fold CV (20 repeats = 100 folds) within the subgroup
rcv100 = RepeatedStratifiedKFold(n_splits=5, n_repeats=20, random_state=RANDOM_STATE)
sc = cross_validate(model_nc, X_nc, y_nc, cv=rcv100, scoring={'acc': 'accuracy', 'bal': 'balanced_accuracy'})
print(f"Subgroup-specific SVC, repeated CV: accuracy {sc['test_acc'].mean():.3f} (SD {sc['test_acc'].std():.3f}), "
      f"balanced accuracy {sc['test_bal'].mean():.3f} (SD {sc['test_bal'].std():.3f})")
pred_nc = model_nc.fit(Xnc_tr, ync_tr).predict(Xnc_te)
print(f"Subgroup-specific SVC, single held-out split (n={len(ync_te)}): accuracy {accuracy_score(ync_te, pred_nc):.3f}, "
      f"balanced accuracy {balanced_accuracy_score(ync_te, pred_nc):.3f}, "
      f"ROC-AUC {roc_auc_score((ync_te == 'Improved').astype(int), p_improved(model_nc, Xnc_te)):.3f}")

## 13. Hospital-level (leave-one-hospital-out) validation

In each iteration all patients from one hospital are held out; the SVC is re-tuned on the other five
hospitals and evaluated on the held-out one (internal-external validation).

In [ ]:
hospital = data.loc[complete_mask, 'place'].reset_index(drop=True)   # recruiting hospital (grouping only)

# How the random split distributed each hospital's patients between training and test sets.
in_train = np.isin(np.arange(len(X)), X_train.index)
save_table(pd.crosstab(hospital, np.where(in_train, 'Train', 'Test')), 'hospital_train_test_distribution.xlsx')


def svc_grid_search(cv_obj, scoring='f1_weighted'):
    """SVC grid search used in the robustness analyses.

    probability=False during the search makes it ~5x faster. This does not change the selected
    hyperparameters, because SVC.predict() (used by the F1 scorer) does not depend on Platt scaling.
    Use with_probability() afterwards to obtain a model that outputs probabilities.
    """
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=False, random_state=RANDOM_STATE))])
    return GridSearchCV(pipe, svc_grid, cv=cv_obj, scoring=scoring, n_jobs=-1)


def with_probability(est):
    """Unfitted copy of an SVC pipeline with Platt-scaled probabilities switched on."""
    e = clone(est); e.set_params(clf__probability=True)
    return e


loho_rows = []
loho_prob = np.zeros(len(X))                         # out-of-hospital predicted P(Improved)
loho_pred = np.empty(len(X), dtype=object)           # out-of-hospital predicted class
loho_prob_pain = np.zeros(len(X))                    # same for the pain-only benchmark (Section 15)
for tr_idx, te_idx in LeaveOneGroupOut().split(X, y, groups=hospital):
    gs = svc_grid_search(cv).fit(X.iloc[tr_idx], y.iloc[tr_idx])            # re-tune on the 5 other hospitals
    m = with_probability(gs.best_estimator_).fit(X.iloc[tr_idx], y.iloc[tr_idx])
    loho_prob[te_idx] = p_improved(m, X.iloc[te_idx]); loho_pred[te_idx] = m.predict(X.iloc[te_idx])
    lp = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE))])
    lp.fit(X.iloc[tr_idx][[PAIN]], y.iloc[tr_idx]); loho_prob_pain[te_idx] = p_improved(lp, X.iloc[te_idx][[PAIN]])
    y_te, p_te = y.iloc[te_idx], loho_pred[te_idx].astype(str)
    lo, hi = proportion_confint((p_te == y_te.values).sum(), len(te_idx), method='wilson')   # 95% CI for accuracy
    loho_rows.append({'Hospital (held out)': hospital.iloc[te_idx[0]], 'N': len(te_idx),
                      '% severe baseline': ceiling[te_idx].mean(),
                      'Accuracy': accuracy_score(y_te, p_te), 'Accuracy 95% CI': f"{lo:.3f}-{hi:.3f}",
                      'F1': f1_score(y_te, p_te, average='weighted'),
                      'Balanced accuracy': balanced_accuracy_score(y_te, p_te), 'MCC': matthews_corrcoef(y_te, p_te),
                      'ROC-AUC': roc_auc_score((y_te == 'Improved').astype(int), loho_prob[te_idx])})
loho_df = pd.DataFrame(loho_rows)
save_table(loho_df, 'leave_one_hospital_out_results.xlsx', index=False)

y_bin_all = (y == 'Improved').astype(int).values
loho_correct = (loho_pred.astype(str) == y.values)
print("Pooled LOHO: accuracy %.3f, balanced accuracy %.3f, MCC %.3f, ROC-AUC %.3f" % (
    accuracy_score(y, loho_pred.astype(str)), balanced_accuracy_score(y, loho_pred.astype(str)),
    matthews_corrcoef(y, loho_pred.astype(str)), roc_auc_score(y_bin_all, loho_prob)))
chi2, p_het, dof, _ = stats.chi2_contingency(pd.crosstab(hospital, loho_correct))
print(f"Heterogeneity of accuracy across hospitals: chi2 = {chi2:.2f}, df = {dof}, p = {p_het:.3f}")
loho_df.round(3)

## 14. Repeated cross-validation and statistical comparison of models

Each tuned model is re-evaluated on the same 100 folds (5-fold CV × 20 repeats within the training set).
Paired comparisons with the SVC use the DeLong test (test-set ROC-AUC), the exact McNemar test
(test-set errors) and the Nadeau–Bengio corrected resampled t-test (100 CV folds), with Holm adjustment.

In [ ]:
fold_scores = {}
for name, m in best_models.items():
    fold_scores[name] = cross_validate(m, X_train, y_train, cv=rcv100,
                                       scoring={'acc': 'accuracy', 'auc': 'roc_auc', 'f1': 'f1_weighted'}, n_jobs=-1)
repeated_df = pd.DataFrame([{'Model': n, 'Accuracy mean': s['test_acc'].mean(), 'Accuracy SD': s['test_acc'].std(),
                             'AUC mean': s['test_auc'].mean(), 'AUC SD': s['test_auc'].std(),
                             'F1 mean': s['test_f1'].mean()} for n, s in fold_scores.items()]
                           ).sort_values('F1 mean', ascending=False).reset_index(drop=True)
save_table(repeated_df, 'repeated_cv_100folds.xlsx', index=False)

order = repeated_df['Model'].tolist()
fig, ax = plt.subplots(figsize=(9, 6))
ax.boxplot([fold_scores[m]['test_acc'] for m in order], tick_labels=order, showmeans=True)
ax.set_ylabel('Cross-validated accuracy (5-fold x 20 repeats = 100 folds)')
ax.set_title('Fold-level cross-validation performance distributions')
plt.xticks(rotation=25, ha='right'); plt.tight_layout(); save_figure('repeated_cv_boxplot.png'); plt.show()
repeated_df.round(3)

In [ ]:
def delong_test(y_true, p1, p2):
    """Two-sided DeLong test for two correlated ROC-AUCs (fast algorithm of Sun & Xu, 2014).

    Returns (auc1, auc2), z statistic, p-value. y_true must be 0/1.
    """
    def midrank(x):
        order_ = np.argsort(x); z = x[order_]; n = len(x); t = np.zeros(n); i = 0
        while i < n:
            j = i
            while j < n and z[j] == z[i]:
                j += 1
            t[i:j] = 0.5 * (i + j - 1) + 1          # average rank for ties
            i = j
        out = np.empty(n); out[order_] = t
        return out
    y_true = np.asarray(y_true).astype(int)
    o = np.argsort(-y_true); m = y_true.sum(); n = len(y_true) - m
    preds = np.vstack([np.asarray(p1)[o], np.asarray(p2)[o]])
    tx = np.array([midrank(r[:m]) for r in preds]); ty = np.array([midrank(r[m:]) for r in preds])
    tz = np.array([midrank(r) for r in preds])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    s = np.cov(v01) / m + np.cov(v10) / n
    z = (aucs[0] - aucs[1]) / np.sqrt(s[0, 0] + s[1, 1] - 2 * s[0, 1])
    return aucs, z, 2 * stats.norm.sf(abs(z))


def corrected_resampled_ttest(a, b, n_train, n_test):
    """Nadeau & Bengio (2003) t-test for paired scores from repeated k-fold CV.

    The variance is inflated by (1/J + n_test/n_train) to account for overlapping training sets.
    """
    d = np.asarray(a) - np.asarray(b); J = len(d)
    t = d.mean() / np.sqrt((1 / J + n_test / n_train) * np.var(d, ddof=1))
    return d.mean(), 2 * stats.t.sf(abs(t), df=J - 1)


svc_correct = (y_pred == y_test.values)
n_tr, n_te = len(X_train) * 4 / 5, len(X_train) / 5          # training/validation sizes within each CV fold
cmp_rows = []
for name, m in best_models.items():
    if name == 'SVC':
        continue
    other_correct = (m.predict(X_test) == y_test.values)
    (auc_svc, auc_other), _, p_delong = delong_test(y_test_bin, y_prob_pos, p_improved(m, X_test))
    table2x2 = [[np.sum(svc_correct & other_correct), np.sum(svc_correct & ~other_correct)],
                [np.sum(~svc_correct & other_correct), np.sum(~svc_correct & ~other_correct)]]
    d_acc, p_acc = corrected_resampled_ttest(fold_scores['SVC']['test_acc'], fold_scores[name]['test_acc'], n_tr, n_te)
    _, p_auc = corrected_resampled_ttest(fold_scores['SVC']['test_auc'], fold_scores[name]['test_auc'], n_tr, n_te)
    cmp_rows.append({'Comparator': name, 'Test AUC': auc_other, 'DeLong p': p_delong,
                     'Errors SVC only': table2x2[1][0], 'Errors comparator only': table2x2[0][1],
                     'McNemar p': mcnemar(table2x2, exact=True).pvalue,
                     'Repeated-CV accuracy difference': d_acc, 'Accuracy p': p_acc, 'AUC p': p_auc})
cmp_df = pd.DataFrame(cmp_rows)
for col in ['DeLong p', 'McNemar p', 'Accuracy p', 'AUC p']:      # Holm correction across the 5 comparisons
    cmp_df[col + ' (Holm)'] = multipletests(cmp_df[col], method='holm')[1]
save_table(cmp_df, 'model_statistical_comparisons.xlsx', index=False)
cmp_df.round(3)

## 15. Pain-only benchmarks and the subgroup without severe baseline pain

The SVC is compared with two models that use preoperative pain only: (a) a deterministic *ceiling rule*
(predict Improved only if baseline pain is severe) and (b) logistic regression with pain as the single
predictor. Results for the subgroup without severe baseline pain refer here to the **full-cohort** model
evaluated in subgroup patients (analysis (i) in the article).

In [ ]:
def binary_summary(yt, yp, prob=None):
    """Accuracy, balanced accuracy and (optionally) ROC-AUC with 'Improved' as positive class."""
    out = {'Accuracy': accuracy_score(yt, yp), 'Balanced accuracy': balanced_accuracy_score(yt, yp)}
    if prob is not None and len(set(yt)) > 1:
        out['ROC-AUC'] = roc_auc_score((np.asarray(yt) == 'Improved').astype(int), prob)
    return out


rule_pred = np.where(X_test[PAIN] == 4, 'Improved', 'Not Improved')          # (a) ceiling rule
lr_pain = Pipeline([('sc', StandardScaler()),
                    ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE))]).fit(X_train[[PAIN]], y_train)
p_pain_test = p_improved(lr_pain, X_test[[PAIN]])                            # (b) pain-only model

bench = pd.DataFrame({
    'Ceiling rule': binary_summary(y_test, rule_pred, (X_test[PAIN] == 4).astype(int)),
    'Pain-only logistic regression': binary_summary(y_test, lr_pain.predict(X_test[[PAIN]]), p_pain_test),
    'SVC (37 predictors)': binary_summary(y_test, y_pred, y_prob_pos)}).T
print(bench.round(3))
_, _, p = delong_test(y_test_bin, y_prob_pos, p_pain_test)
print(f"DeLong test, SVC vs pain-only (test set): p = {p:.3f}")
_, _, p = delong_test(y_bin_all, loho_prob, loho_prob_pain)
print(f"LOHO ROC-AUC: SVC {roc_auc_score(y_bin_all, loho_prob):.3f} vs pain-only {roc_auc_score(y_bin_all, loho_prob_pain):.3f} (DeLong p = {p:.3f})")

# Full-cohort SVC evaluated only in test patients without severe baseline pain.
nc_test = (X_test[PAIN] != 4).values
sub = binary_summary(y_test[nc_test], y_pred[nc_test], y_prob_pos[nc_test])
print(f"\nSubgroup without severe pain in the test set (n={nc_test.sum()}):",
      {k: round(v, 3) for k, v in sub.items()},
      f"| majority-class accuracy = {(y_test[nc_test] == 'Not Improved').mean():.3f}",
      f"| predicted Not Improved for {(y_pred[nc_test] == 'Not Improved').sum()} patients")
print(f"Subgroup ROC-AUC under LOHO: {roc_auc_score(y_bin_all[~ceiling], loho_prob[~ceiling]):.3f}")

## 16. Decision-threshold analysis

The clinically actionable event is **Not Improved** (poor pain recovery). Two thresholds are derived
from training-set out-of-fold predictions only, then applied unchanged to the test set.

In [ ]:
def event_metrics(y_event, flag):
    """Sensitivity, specificity, PPV, NPV for a 0/1 event vector and 0/1 flag vector."""
    tp_ = np.sum(flag & y_event); fn_ = np.sum(~flag & y_event)
    fp_ = np.sum(flag & ~y_event); tn_ = np.sum(~flag & ~y_event)
    return {'Sensitivity': tp_ / (tp_ + fn_), 'Specificity': tn_ / (tn_ + fp_),
            'PPV': tp_ / (tp_ + fp_) if tp_ + fp_ else np.nan, 'NPV': tn_ / (tn_ + fn_) if tn_ + fn_ else np.nan,
            'Missed at-risk': int(fn_), 'Flagged': int(tp_ + fp_)}


# Out-of-fold P(Not Improved) in the training set with the final SVC configuration.
oof = cross_val_predict(clone(best_model), X_train, y_train, cv=cv, method='predict_proba')
oof_not = oof[:, list(best_model.classes_).index('Not Improved')]
y_tr_not = (y_train == 'Not Improved').values
grid_t = np.round(np.arange(0.05, 0.951, 0.01), 2)
train_metrics = [event_metrics(y_tr_not, oof_not >= t) for t in grid_t]
t_youden = grid_t[np.argmax([m['Sensitivity'] + m['Specificity'] - 1 for m in train_metrics])]
t_sens90 = grid_t[np.where([m['Sensitivity'] >= 0.90 for m in train_metrics])[0].max()]

y_not_bool = y_not_test.astype(bool)
rows = {'Default SVM decision boundary': event_metrics(y_not_bool, y_pred == 'Not Improved')}
for label, t in [('0.30', 0.30), (f'Youden (training) = {t_youden:.2f}', t_youden),
                 (f'>=90% sensitivity (training) = {t_sens90:.2f}', t_sens90),
                 ('0.40', 0.40), ('0.50', 0.50), ('0.60', 0.60)]:
    rows[f'Threshold {label}'] = event_metrics(y_not_bool, p_not_test >= t)
threshold_df = pd.DataFrame(rows).T
save_table(threshold_df, 'threshold_analysis.xlsx')
threshold_df.round(3)

## 17. Correlated predictors and domain-level importance

In [ ]:
# Pairs of predictors with |Spearman rho| >= 0.70 in the training set (constant columns give NaN -> 0).
corr = X_train.corr(method='spearman').fillna(0)
cols = list(X.columns)
pairs = [(feature_display_names[cols[i]], feature_display_names[cols[j]], round(float(corr.iloc[i, j]), 3))
         for i in range(len(cols)) for j in range(i + 1, len(cols)) if abs(corr.iloc[i, j]) >= 0.7]
print("Strongly correlated pairs:", *sorted(pairs, key=lambda r: -abs(r[2])), sep='\n  ')

# Eight clinical domains; correlated predictors fall in the same domain.
domains = {
    'Preoperative pain': [PAIN],
    'Patient psychological state': ['Depresion_pre', 'Ansiedad_pre', 'Estres_pre'],
    'Fatigue (FSMC)': ['prefatiguecognitivescale', 'prefatiguephysicalscale', 'preFSMCtotal'],
    'Functional status and fitness': ['preselfcare', 'preSphinctercontrol', 'pretransfer', 'prelocomtion',
                                      'PRPMotorScore', 'precommunication', 'presocialcogintive',
                                      'PRPCognitiveScore', 'PRPTOTALFIM', 'prekps', 'preTotalIFIS'],
    'Comorbidity and medication': ['HTN', 'Medication_reccd', 'precharolsonindex', 'AgeCharlson'],
    'Lifestyle': ['Smoke', 'Alcohol'],
    'Demographic and socioeconomic': ['AGE_years', 'wieght', 'Height', 'gender', 'MaritalStatus',
                                      'numberofchilderen', 'educationlevel', 'occupation', 'economicproblems'],
    'Family functioning and caregiver psychosocial': ['preapgar', 'prefamiliaestres',
                                                      'prefamiliaresdepreseion', 'preansiedadfamilia'],
}
assert sorted(sum(domains.values(), [])) == sorted(cols)     # every predictor belongs to exactly one domain

# Grouped permutation importance: permute all columns of a domain together (same row order), 100 times.
base_f1, base_auc = f1_score(y_test, y_pred, average='weighted'), auc_value
perm_rng = np.random.RandomState(RANDOM_STATE)   # dedicated generator so results do not depend on earlier cells
dom_rows = []
for dname, dcols in domains.items():
    d_f1, d_auc = [], []
    for _ in range(100):
        Xp = X_test.copy(); perm_idx = perm_rng.permutation(len(Xp))
        Xp[dcols] = Xp[dcols].values[perm_idx]
        d_f1.append(base_f1 - f1_score(y_test, best_model.predict(Xp), average='weighted'))
        d_auc.append(base_auc - roc_auc_score(y_test_bin, p_improved(best_model, Xp)))
    dom_rows.append({'Domain': dname, 'N predictors': len(dcols), 'Permutation dF1': np.mean(d_f1),
                     'Permutation dAUC': np.mean(d_auc)})

# Grouped SHAP: average three Kernel SHAP runs (different seeds, 500 samples) for stability,
# then sum SHAP values within each domain for each patient.
shap_runs = []
for seed in [0, 1, 2]:
    ex = shap.KernelExplainer(predict_improved_proba, shap.sample(X_train, 30, random_state=seed))
    np.random.seed(seed)
    shap_runs.append(np.asarray(ex.shap_values(X_test, nsamples=500, silent=True)))
print("SHAP run-to-run rank correlation:",
      [round(float(spearmanr(np.abs(shap_runs[i]).mean(0), np.abs(shap_runs[j]).mean(0))[0]), 3) for i, j in [(0, 1), (0, 2), (1, 2)]])
shap_avg = np.mean(shap_runs, axis=0)
for r in dom_rows:
    idx_cols = [cols.index(c) for c in domains[r['Domain']]]
    r['Grouped |SHAP|'] = np.abs(shap_avg[:, idx_cols].sum(axis=1)).mean()
domain_df = pd.DataFrame(dom_rows)
rho, p = spearmanr(domain_df['Permutation dAUC'], domain_df['Grouped |SHAP|'])
print(f"Domain level: permutation dAUC vs grouped SHAP, Spearman rho = {rho:.2f} (p = {p:.3f})")
save_table(domain_df, 'domain_importance.xlsx', index=False)
domain_df.round(3)

## 18. Overfitting and stability

* **Nested CV** (outer 5-fold × `N_NESTED_REPEATS`, inner 5-fold grid search): the tuning step is
  repeated without seeing the outer validation fold.
* **Nested CV of the full procedure**: algorithm *selection* among the six models is also repeated in
  every inner loop (quantifies optimism from choosing the model after seeing test results).
* **Label-permutation test** and **learning curve**.

In [ ]:
def nested_cv(Xd, yd, model='svc', n_repeats=N_NESTED_REPEATS, scoring='f1_weighted'):
    """Repeated nested CV. Returns per-repeat out-of-fold P(first class) and predicted labels.

    model='svc' re-tunes the SVC grid inside each outer training fold; otherwise the given
    estimator is simply refitted (used for the pain-only model, which has nothing to tune).
    """
    outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats, random_state=RANDOM_STATE)
    probs = np.full((n_repeats, len(yd)), np.nan); preds = np.empty((n_repeats, len(yd)), dtype=object)
    for k, (tr, te) in enumerate(outer.split(Xd, yd)):
        inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        if model == 'svc':
            est = with_probability(svc_grid_search(inner, scoring).fit(Xd.iloc[tr], yd.iloc[tr]).best_estimator_)
        else:
            est = clone(model)
        est.fit(Xd.iloc[tr], yd.iloc[tr])
        probs[k // 5, te] = est.predict_proba(Xd.iloc[te])[:, 0]     # column 0 = alphabetically first class
        preds[k // 5, te] = est.predict(Xd.iloc[te])
    return probs, preds


def summarize_nested(probs, preds, yd, positive, mask=None):
    """Mean (SD) over repeats of pooled out-of-fold ROC-AUC, accuracy and balanced accuracy."""
    m = np.ones(len(yd), bool) if mask is None else mask
    yt = yd.values[m]
    auc_ = [roc_auc_score((yt == positive).astype(int), p[m]) for p in probs]
    acc_ = [accuracy_score(yt, q[m].astype(str)) for q in preds]
    bal_ = [balanced_accuracy_score(yt, q[m].astype(str)) for q in preds]
    return {k: f"{np.mean(v):.3f} ({np.std(v):.3f})" for k, v in [('ROC-AUC', auc_), ('Accuracy', acc_), ('Balanced accuracy', bal_)]}


pain_only_lr = Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE))])
if RUN_SLOW_ANALYSES:
    reduced_drop = ['Alcohol', 'PRPMotorScore', 'PRPCognitiveScore', 'PRPTOTALFIM', 'preFSMCtotal', 'AgeCharlson']
    nested = {'SVC, 37 predictors': nested_cv(X, y),
              'SVC, no preoperative pain': nested_cv(X.drop(columns=[PAIN]), y),
              'SVC, reduced 31 predictors': nested_cv(X.drop(columns=reduced_drop), y),
              'Pain-only logistic regression': nested_cv(X[[PAIN]], y, model=pain_only_lr)}
    nested_df = pd.DataFrame({k: summarize_nested(*v, y, 'Improved') for k, v in nested.items()}).T
    nested_sub = pd.DataFrame({k: summarize_nested(*nested[k], y, 'Improved', mask=~ceiling)
                               for k in ['SVC, 37 predictors', 'Pain-only logistic regression']}).T
    print("Nested CV, all 192 patients - mean (SD over repeats):\n", nested_df)
    print("\nNested CV restricted to patients without severe baseline pain:\n", nested_sub)
    # Calibration slope of the nested out-of-fold risk of not improving, averaged over the repeats
    # (slope = 1 means predictions are neither too extreme nor too modest).
    slopes = []
    for p_imp_rep in nested['SVC, 37 predictors'][0]:
        p_not_oof = np.clip(1 - p_imp_rep, 1e-6, 1 - 1e-6)
        lp = np.log(p_not_oof / (1 - p_not_oof))                      # logit of the predicted risk
        slopes.append(sm.Logit(1 - y_bin_all, sm.add_constant(lp)).fit(disp=0).params[1])
    print("Calibration slope (nested OOF, mean over repeats):", round(np.mean(slopes), 2))
    print("Apparent training accuracy of the final SVC:", round(accuracy_score(y_train, best_model.predict(X_train)), 3))

In [ ]:
# Nested CV of the full procedure: the inner loop selects the best of the six tuned algorithms.
if RUN_SLOW_ANALYSES:
    outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
    sel_rows = []
    for k, (tr, te) in enumerate(outer.split(X, y)):
        inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        best_name, best_score, best_est = None, -1, None
        for name, (est, grid) in model_grids.items():
            gs = svc_grid_search(inner) if name == 'SVC' else GridSearchCV(est, grid, cv=inner, scoring='f1_weighted', n_jobs=-1)
            gs.fit(X.iloc[tr], y.iloc[tr])
            if gs.best_score_ > best_score:
                best_name, best_score, best_est = name, gs.best_score_, gs.best_estimator_
        est = with_probability(best_est) if best_name == 'SVC' else clone(best_est)
        est.fit(X.iloc[tr], y.iloc[tr])
        pr = est.predict(X.iloc[te])
        sel_rows.append({'Outer fold': k, 'Selected model': best_name, 'Accuracy': accuracy_score(y.iloc[te], pr),
                         'Balanced accuracy': balanced_accuracy_score(y.iloc[te], pr),
                         'ROC-AUC': roc_auc_score((y.iloc[te] == 'Improved').astype(int), p_improved(est, X.iloc[te]))})
    sel_df = pd.DataFrame(sel_rows)
    print(sel_df[['Accuracy', 'Balanced accuracy', 'ROC-AUC']].agg(['mean', 'std']).round(3))
    print("Algorithm selected in the inner loop:", sel_df['Selected model'].value_counts().to_dict())
    save_table(sel_df, 'nested_cv_model_selection.xlsx', index=False)

In [ ]:
# Label-permutation tests and learning curve use the fixed final SVC configuration.
final_svc_fast = clone(best_model).set_params(clf__probability=False)    # probabilities not needed here
if RUN_SLOW_ANALYSES:
    score, null, p = permutation_test_score(final_svc_fast, X, y, cv=cv5, scoring='roc_auc',
                                            n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE, n_jobs=-1)
    print(f"All patients: CV ROC-AUC = {score:.3f}, null mean = {null.mean():.3f}, "
          f"null 95th pct = {np.percentile(null, 95):.3f}, p = {p:.4f}")
    for metric in ['roc_auc', 'balanced_accuracy']:
        score, null, p = permutation_test_score(final_svc_fast, X_nc, y_nc, cv=cv5, scoring=metric,
                                                n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE, n_jobs=-1)
        print(f"Subgroup without severe pain: CV {metric} = {score:.3f}, p = {p:.4f}")

    rcv50 = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
    for ax, metric, label in [(axs[0], 'roc_auc', 'ROC-AUC'), (axs[1], 'accuracy', 'Accuracy')]:
        sizes, tr_s, va_s = learning_curve(final_svc_fast, X, y, cv=rcv50, scoring=metric,
                                           train_sizes=np.linspace(0.2, 1.0, 9), n_jobs=-1, random_state=RANDOM_STATE)
        for scores, lab, mk in [(tr_s, 'Training folds', 'o-'), (va_s, 'Validation folds', 's-')]:
            ax.plot(sizes, scores.mean(1), mk, label=lab)
            ax.fill_between(sizes, scores.mean(1) - scores.std(1), scores.mean(1) + scores.std(1), alpha=0.15)
        ax.set_xlabel('Number of training patients'); ax.set_ylabel(label); ax.grid(alpha=0.3); ax.legend(loc='lower right')
        ax.set_title(f'Learning curve: {label}')
    plt.tight_layout(); save_figure('learning_curve.png'); plt.show()

## 19. Alternative outcome and leave-one-hospital-out calibration / decision curves

In [ ]:
# Alternative outcome not calculated from baseline pain: moderate or severe pain at follow-up.
y_alt = pd.Series(np.where(post_cat >= 3, 'Moderate/severe', 'None/mild'))
print("Alternative outcome:", y_alt.value_counts().to_dict())
if RUN_SLOW_ANALYSES:
    alt = {'SVC, 37 predictors': nested_cv(X, y_alt, scoring='roc_auc'),
           'SVC, no preoperative pain': nested_cv(X.drop(columns=[PAIN]), y_alt, scoring='roc_auc'),
           'Pain-only logistic regression': nested_cv(X[[PAIN]], y_alt, model=pain_only_lr)}
    print(pd.DataFrame({k: summarize_nested(*v, y_alt, 'Moderate/severe') for k, v in alt.items()}).T)

In [ ]:
# Calibration and decision curve using the pooled LOHO predictions for all 192 patients (Section 13).
p_not_loho = 1 - loho_prob
y_not_all = 1 - y_bin_all
lp = np.log(np.clip(p_not_loho, 1e-6, 1 - 1e-6) / (1 - np.clip(p_not_loho, 1e-6, 1 - 1e-6)))
slope = sm.Logit(y_not_all, sm.add_constant(lp)).fit(disp=0).params[1]
intercept = sm.GLM(y_not_all, np.ones_like(lp), offset=lp, family=sm.families.Binomial()).fit().params[0]
print(f"LOHO calibration: slope = {slope:.2f}, intercept = {intercept:.2f}, Brier = {brier_score_loss(y_not_all, p_not_loho):.3f}")

th = np.round(np.arange(0.05, 0.951, 0.025), 3)
prev_all = y_not_all.mean()
nb_model = np.array([net_benefit(y_not_all, p_not_loho, t) for t in th])
nb_all = prev_all - (1 - prev_all) * th / (1 - th)
nb_b = []
boot_rng = np.random.RandomState(RANDOM_STATE)   # dedicated generator for this bootstrap
for _ in range(1000):
    b = boot_rng.randint(0, len(y_not_all), len(y_not_all))
    nb_b.append([net_benefit(y_not_all[b], p_not_loho[b], t) for t in th])
lo, hi = np.percentile(nb_b, 2.5, axis=0), np.percentile(nb_b, 97.5, axis=0)
ok = (lo > nb_all) & (lo > 0)
print("LOHO: lower 95% bound beats both defaults from", th[ok].min(), "to", th[ok].max())

fp_l, mp_l = calibration_curve(y_not_all, p_not_loho, n_bins=8, strategy='quantile')
fig, axs = plt.subplots(1, 2, figsize=(13, 5.6))
axs[0].plot(mp_l, fp_l, 'o-', label='SVC, pooled LOHO'); axs[0].plot([0, 1], [0, 1], '--', color='gray', label='Perfect calibration')
axs[0].set_xlabel('Mean predicted probability of not improving'); axs[0].set_ylabel('Observed fraction not improved')
axs[0].set_title('(a) Calibration, leave-one-hospital-out'); axs[0].legend(loc='upper left'); axs[0].grid(alpha=.3)
axs[1].plot(th, nb_model, lw=2, label='SVC (pooled LOHO)'); axs[1].fill_between(th, lo, hi, alpha=.2, label='95% bootstrap CI')
axs[1].plot(th, nb_all, '--', label='Treat all'); axs[1].axhline(0, ls=':', color='C2', label='Treat none')
axs[1].set_ylim(-0.1, 0.6); axs[1].set_xlabel('Threshold probability'); axs[1].set_ylabel('Net benefit')
axs[1].set_title('(b) Decision curve, leave-one-hospital-out'); axs[1].legend(loc='upper right'); axs[1].grid(alpha=.3)
plt.tight_layout(); save_figure('loho_calibration_dca.png'); plt.show()

## 20. External validation helper and reproducibility summary

In [ ]:
def evaluate_on_external_dataset(external_path, trained_model=best_model):
    """Apply the trained final model to an external dataset with the same columns as Data.xlsx.

    The external file must contain the 37 predictor columns (same names and text codings, see
    feature_dictionary.csv) and the 'pain_change' outcome column. Rows with missing predictors
    are dropped, mirroring the complete-case analysis.
    """
    ext = pd.read_excel(external_path)
    X_ext = encode_features(ext[feature_columns])
    keep = X_ext.notna().all(axis=1)
    X_ext, y_ext = X_ext[keep], to_outcome_class(ext.loc[keep, 'pain_change'])
    pred_ext = trained_model.predict(X_ext)
    print(classification_report(y_ext, pred_ext))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_ext, pred_ext), 3))
    print("ROC-AUC:", round(roc_auc_score((y_ext == 'Improved').astype(int), p_improved(trained_model, X_ext)), 3))

# Example (uncomment when an external dataset is available):
# evaluate_on_external_dataset('external_hospital_data.xlsx')

In [ ]:
print("REPRODUCIBILITY SUMMARY")
print("Final model:", best_model)
print("Random seed:", RANDOM_STATE, "| split: 70/30 stratified | tuning CV: stratified 5-fold, weighted F1")
print("Training patients:", len(X_train), "| test patients:", len(X_test))